In [18]:
# Importar bibliotecas e iniciar o Spark

import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr

spark = (
    SparkSession
    .builder
    .appName("ETL-Netflix")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)

print("Bibliotecas importadas e spark iniciado")

Bibliotecas importadas e spark iniciado


In [10]:
# Definir diretórios

base_path = "../input"
bronze_path = os.path.join(base_path, "bronze")
base_csv_path = os.path.join(base_path, "csv", "netflix_titles.csv")

os.makedirs(bronze_path, exist_ok=True)

print("Diretórios definidos")

Diretórios definidos


In [14]:
# Criar o dataframe inicial

df_base = spark.read.csv(
    base_csv_path,
    header=True,
    inferSchema=True,
    )

df_base.withColumn(
    "release_year",
    col("release_year").cast("int")
)

df_base = df_base.filter(col("release_year") >= 2000)

print("Dataframe criado:")
df_base.show(5)

Dataframe criado:
+-------+-------+--------------------+---------------+--------------------+-------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|       director|                cast|      country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+---------------+--------------------+-------------+------------------+------------+------+---------+--------------------+--------------------+
|     s1|  Movie|Dick Johnson Is Dead|Kirsten Johnson|                NULL|United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|
|     s2|TV Show|       Blood & Water|           NULL|Ama Qamata, Khosi...| South Africa|September 24, 2021|        2021| TV-MA|2 Seasons|International TV ...|After crossing pa...|
|     s3|TV Show|           Ganglands|Julien Leclercq|Sami Bouajila, Tr...|  

In [17]:
# Salvando os dados extraidos na camada bronze

bronze_output_path = os.path.join(bronze_path, "netflix-bronze")

df_base.write.mode("overwrite").option("header", True).csv(bronze_output_path)

print(f"Arquivo salvo com sucesso em: {bronze_output_path}")

{"ts": "2025-11-05 01:19:48.096", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value ' Paul Sambo' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "line 14 in cell [14]", "line": "", "fragment": "__ge__", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o117.csv.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value ' Paul Sambo' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"__ge__\" was called from\nline 14 in cell [14]\n\r\n\tat org.apache.spark.sql.errors

NumberFormatException: [CAST_INVALID_INPUT] The value ' Paul Sambo' of the type "STRING" cannot be cast to "BIGINT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"__ge__" was called from
line 14 in cell [14]


In [ ]:
# Parando o spark

spark.stop()

print("Sessão Spark foi encerrada")